# **Task 3: Build a Chatbot using Hugging Face Transformers**

In [6]:
!pip install "transformers>=4.37.0" accelerate

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!


In [8]:
def generate_response(messages):
    # Converting the chat messages into a single formatted prompt
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Converting the text into tokens
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generating the response using the transformer model
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Removing the input tokens and keep only newly generated tokens
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Converting the generated tokens back to readable text
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Returning the final chatbot response
    return response

In [11]:
def chatbot():
    """
    Console-based chatbot function
    Call chatbot() to start interaction
    """
    
    # Initial default message
    print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")
    
    # Initializing conversation history
    messages = [
        {"role": "system", "content": "You are a helpful and friendly AI assistant."}
    ]
    
    while True:
        # Taking the user input
        user_input = input("You: ")
        
        # applyong the exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break
        
        # Adding user message to history
        messages.append({
            "role": "user",
            "content": user_input
        })
        
        # Generating the response from the model
        response = generate_response(messages)
        
        # Printing the chatbot response
        print("Chatbot:", response)
        
        # Adding all the bot response to history
        messages.append({
            "role": "assistant",
            "content": response
        })

In [12]:
chatbot()

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.


You:  Hello


Chatbot: Hello! How can I assist you today?


You:  What is Artificial Intelligence?


Chatbot: Artificial Intelligence (AI) refers to the development of computer systems that exhibit intelligent behavior, such as learning, problem-solving, decision-making, perception, and natural language processing. These systems are designed to perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision making in complex environments, and generating human-like responses.

AI has revolutionized many fields, including healthcare, finance,


...

Chatbot: Python was created byGuido van Rossum in 1989 at Utrecht University in The Netherlands. Guido van Rossum was the creator of the Python programming language. He started developing the language in his free time during the late 1970s and early 1980s while he worked on a computer science project at Utrecht University.

Python's popularity grew


You:  Thank you


Chatbot: You're welcome! If you have any more questions, feel free to ask.


You:  exit


Chatbot: Goodbye! Have a great day.
